In [1]:
from tests.data.test_fate_anndata import TestFateAnnData


h5ad_filename = "test_write_h5ad.h5ad"

[2025年05月21日 17时52分03秒] INFO                                                                                 
                                          _____     _ _ ______    _       ______            _                      
                                         / ____|   | | |  ____|  | |     |  ____|          | |                     
                                        | |     ___| | | |__ __ _| |_ ___| |__  __  ___ __ | | ___  _ __ ___ _ __  
                                        | |    / _ \ | |  __/ _` | __/ _ \  __| \ \/ / '_ \| |/ _ \| '__/ _ \ '__| 
                                        | |___|  __/ | | | | (_| | ||  __/ |____ >  <| |_) | | (_) | | |  __/ |    
                                         \_____\___|_|_|_|  \__,_|\__\___|______/_/\_\ .__/|_|\___/|_|  \___|_|    
                                                                                     | |                           
                                                                              

测试用例中添加了MilestoneWrapper和WaypoitnWrapperFateAnnData对象

In [2]:
test_fate_anndata = TestFateAnnData()

test_fate_anndata.setup_method()
test_fate_anndata.test_add_waypoints()
fadata = test_fate_anndata.fadata

fadata

AnnData object with n_obs × n_vars = 6 × 2
    obs: 'clusters'
    uns: 'cfe'
    obsm: 'X_emb'
    layers: 'counts', 'expression'

In [3]:
fadata.write_h5ad(h5ad_filename)

transfer 'default' to dict


In [4]:
import h5py

def h5_to_dict(h5_file, path="/"):
    """递归读取 HDF5 文件并转换为嵌套字典"""
    result = {}
    
    # 遍历当前路径下的所有 key
    for key in h5_file[path].keys():
        item_path = f"{path}/{key}" if path != "/" else f"/{key}"
        
        # 如果是 Group（类似文件夹），递归处理
        if isinstance(h5_file[item_path], h5py.Group):
            result[key] = h5_to_dict(h5_file, item_path)
        
        # 如果是 Dataset（数据），读取数据
        elif isinstance(h5_file[item_path], h5py.Dataset):
            result[key] = h5_file[item_path][()]  # [()] 读取全部数据
        
    return result

# 读取 .h5ad 文件并转换为字典
def read_h5ad_to_dict(file_path):
    with h5py.File(file_path, "r") as f:
        return h5_to_dict(f)

# 示例：读取 .h5ad 文件
data_dict = read_h5ad_to_dict(h5ad_filename)

data_dict

{'X': {'data': array([ 8, 12, 20, 15, 22, 10, 10, 12, 20, 16, 20]),
  'indices': array([1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], dtype=int32),
  'indptr': array([ 0,  5, 11], dtype=int32)},
 'layers': {'counts': {'data': array([ 8, 12, 20, 15, 22, 10, 10, 12, 20, 16, 20]),
   'indices': array([1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], dtype=int32),
   'indptr': array([ 0,  5, 11], dtype=int32)},
  'expression': {'data': array([ 8, 12, 20, 15, 22, 10, 10, 12, 20, 16, 20]),
   'indices': array([1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], dtype=int32),
   'indptr': array([ 0,  5, 11], dtype=int32)}},
 'obs': {'_index': array([b'a', b'b', b'c', b'd', b'e', b'f'], dtype=object),
  'clusters': array([1, 1, 2, 2, 2, 3])},
 'obsm': {'X_emb': array([[ 0, 10],
         [ 8, 10],
         [12, 12],
         [20, 20],
         [15, 16],
         [22, 20]])},
 'obsp': {},
 'uns': {'cfe': {'prior_information': {},
   'trajectory_history_dict': {'default': {'milestone_wrapper': {'cell_id_list': array([b'a', b'b', b'c', b'd',

In [5]:
fadata.uns["cfe"]["trajectory_history_dict"]["default"]["milestone_wrapper"]["milestone_network"]

,from,to,length,directed
0,W,X,1.0,True
1,X,Y,1.0,True
2,X,Z,1.0,True
3,Z,A,2.0,True


In [6]:
import scanpy as sc

fadata =  sc.read(h5ad_filename)
fadata

AnnData object with n_obs × n_vars = 6 × 2
    obs: 'clusters'
    uns: 'cfe'
    obsm: 'X_emb'
    layers: 'counts', 'expression'

In [8]:
mw = fadata.uns["cfe"]["trajectory_history_dict"]["default"]["milestone_wrapper"]
mw["milestone_network"]

,from,to,length,directed
0,W,X,1.0,True
1,X,Y,1.0,True
2,X,Z,1.0,True
3,Z,A,2.0,True


尝试在另一个新的环境单独导入下读取该h5ad文件，会成功